<div dir="rtl">

# نوت‌بوک ۶: امکان‌پذیری بازه‌های زمانی

**فاز ۱، تسک ۱۰**

واحد گزارش پروژه «**یک محصول در یک بازه‌ی زمانی**» است (پیش‌فرض ۳ ماه). برای هر واحد، از روی نقدهای همان بازه به صاحب محصول می‌گوییم کدام جنبه‌ها را اول بهبود دهد. این نوت‌بوک می‌پرسد:

۱. چند «محصول در یک بازه» نقد **کافی** دارند؟ کافی یعنی چند نقد؟
۲. چون گزارش درباره‌ی **شکایت‌ها**ست، چند واحد **نقد منفی** کافی هم دارند؟
۳. اگر طول بازه را ۱ یا ۶ ماه بگیریم، این اعداد چطور عوض می‌شوند؟
۴. بازه‌ی زمانی تحلیل از کجا تا کجا باشد؟

در پایان، **آستانه‌ی حداقل نقد** و **بازه‌ی زمانی تحلیل** را انتخاب و قبل از مدل‌سازی ثبت می‌کنیم.

</div>

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = yaml.safe_load(open(ROOT / "configs" / "config.yaml", encoding="utf-8"))
PROC = ROOT / cfg["data"]["processed_dir"]
FIG = ROOT / "figures"
WINDOW_MONTHS = cfg["window"]["months"]

pd.set_option("display.width", 200)
BAR, NEG = "#3B6EA8", "#B5523B"
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e6e6", "grid.linewidth": 0.6,
    "axes.axisbelow": True, "font.size": 9,
})

reviews = (pd.read_parquet(PROC / "reviews_rc.parquet",
                           columns=["parent_asin", "user_id", "rating", "timestamp", "text"])
             .drop_duplicates().reset_index(drop=True))
reviews["date"] = pd.to_datetime(reviews["timestamp"], unit="ms")
reviews["negative"] = reviews["rating"] <= 2
print(f"reviews: {len(reviews):,}   window length from config: {WINDOW_MONTHS} months")

<div dir="rtl">

## ۱. ساختن بازه‌ها

بازه‌ها **تقویمی** و هم‌طول‌اند: برای ۳ ماه یعنی ژانویه تا مارس، آوریل تا ژوئن و ... . بازه‌ی تقویمی دو مزیت دارد: برای همه‌ی محصولات یکسان است، و صاحب محصول هم معمولاً فصلی برنامه‌ریزی می‌کند.

تابع زیر طول بازه را به‌عنوان ورودی می‌گیرد تا بتوانیم ۱ و ۶ ماه را هم امتحان کنیم.

</div>

In [ ]:
def add_window(df, months):
    month_index = df["date"].dt.year * 12 + df["date"].dt.month - 1
    start_index = (month_index // months) * months
    year, month = start_index // 12, start_index % 12 + 1
    return pd.to_datetime(pd.DataFrame({"year": year, "month": month, "day": 1}))

def window_table(df, months):
    w = df.assign(window=add_window(df, months))
    return (w.groupby(["parent_asin", "window"])
             .agg(reviews=("rating", "size"), negatives=("negative", "sum"),
                  mean_rating=("rating", "mean"))
             .reset_index())

units = window_table(reviews, WINDOW_MONTHS)
print(f"product-windows with at least one review: {len(units):,}")
units["reviews"].describe(percentiles=[.5, .75, .9, .95, .99]).round(1)

<div dir="rtl">

## ۲. با چه آستانه‌ای چند واحد داریم؟

برای هر آستانه‌ی «حداقل نقد در بازه» می‌شماریم:

- چند واحد (محصول در یک بازه) واجد شرایط‌اند
- این واحدها از چند **محصول مختلف** می‌آیند
- چند درصد **کل نقدها** در این واحدها هستند

</div>

In [ ]:
total_reviews = len(reviews)
rows = []
for k in [5, 10, 15, 20, 30, 50, 100]:
    ok = units[units["reviews"] >= k]
    rows.append({"min_reviews": k, "units": len(ok), "products": ok["parent_asin"].nunique(),
                 "reviews_covered_%": round(100 * ok["reviews"].sum() / total_reviews, 1),
                 "median_reviews_per_unit": ok["reviews"].median()})
threshold_table = pd.DataFrame(rows).set_index("min_reviews")
threshold_table

<div dir="rtl">

## ۳. نقد منفی کافی

گزارش ما از **شکایت‌ها** ساخته می‌شود. واحدی که ۴۰ نقد دارد ولی همه ۵ ستاره‌اند، چیزی برای اولویت‌بندی ندارد. پس علاوه بر کل نقد، تعداد نقدهای **منفی** (۱ و ۲ ستاره) را هم می‌شماریم.

</div>

In [ ]:
grid = pd.DataFrame(
    {f">= {m} negative": [(units[(units["reviews"] >= k) & (units["negatives"] >= m)]).shape[0]
                          for k in [10, 20, 30, 50]]
     for m in [0, 3, 5, 10]},
    index=pd.Index([10, 20, 30, 50], name="min_reviews"))
grid

<div dir="rtl">

## ۴. حساسیت به طول بازه

همان شمارش را با بازه‌های ۱، ۳ و ۶ ماهه تکرار می‌کنیم. بازه‌ی کوتاه‌تر به‌روزتر است ولی نقد کمتری در هر واحد دارد؛ بازه‌ی بلندتر نقد بیشتری دارد ولی کندتر به تغییرات محصول واکنش نشان می‌دهد.

</div>

In [ ]:
rows = []
for months in [1, 3, 6]:
    u = units if months == WINDOW_MONTHS else window_table(reviews, months)
    for k in [10, 20, 30]:
        ok = u[(u["reviews"] >= k)]
        rows.append({"window_months": months, "min_reviews": k, "units": len(ok),
                     "products": ok["parent_asin"].nunique(),
                     "units_with_>=5_negatives": int((ok["negatives"] >= 5).sum())})
sensitivity = pd.DataFrame(rows).pivot(index="min_reviews", columns="window_months")
sensitivity

<div dir="rtl">

## ۵. واحدهای واجد شرایط در طول زمان

با آستانه‌ی پیشنهادی ۲۰ نقد، تعداد واحدهای واجد شرایط را برای هر سال می‌شماریم. این نشان می‌دهد تحلیل از چه سالی معنادار می‌شود و آیا سال آخر (۲۰۲۳) قابل استفاده است.

</div>

In [ ]:
K = 20
eligible = units[units["reviews"] >= K].copy()
eligible["year"] = eligible["window"].dt.year
per_window = units.assign(eligible=units["reviews"] >= K).groupby("window")["eligible"].sum()
per_year = eligible.groupby("year").agg(units=("reviews", "size"),
                                         products=("parent_asin", "nunique"),
                                         mean_negative_share=("negatives", "sum"))
per_year["mean_negative_share"] = (100 * per_year["mean_negative_share"]
                                   / eligible.groupby("year")["reviews"].sum()).round(1)
per_year.T

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.4))
s = per_window[per_window.index >= "2012-01-01"]
ax.bar(s.index, s.values, width=80, color=BAR, align="edge")
last = reviews["date"].max()
ax.axvspan(pd.Timestamp("2023-04-01"), s.index.max() + pd.Timedelta(days=92),
           color="#f3d9d3", alpha=0.6, lw=0)
ax.text(pd.Timestamp("2023-04-10"), s.max() * 0.9, "from Apr 2023:\nincomplete", fontsize=8, color="#7a3424")
ax.set_ylabel(f"products with >= {K} reviews")
ax.set_title(f"Report-ready product-windows per {WINDOW_MONTHS}-month window", loc="left")
plt.tight_layout()
plt.savefig(FIG / "fig14_eligible_windows.png", dpi=200, bbox_inches="tight")
plt.show()

<div dir="rtl">

## ۶. بازه‌ی تقویمی ناقص آخر

آخرین نقد داده در شهریور ۲۰۲۳ است. بازه‌ای که هنوز تمام نشده بود، به‌صورت ساختاری نقد کمتری دارد. سهم نقدهای هر بازه‌ی ۲۰۲۳ را با همان بازه در ۲۰۲۲ مقایسه می‌کنیم:

</div>

In [ ]:
q = reviews.assign(window=add_window(reviews, WINDOW_MONTHS)).groupby("window").size()
compare = pd.DataFrame({"2022": q[q.index.year == 2022].values[:4],
                        "2023": list(q[q.index.year == 2023].values) + [0] * (4 - (q.index.year == 2023).sum())},
                       index=[f"window {i + 1}" for i in range(4)])
compare["2023 as % of 2022"] = (100 * compare["2023"] / compare["2022"]).round(0)
compare

<div dir="rtl">

## ۷. پیشنهاد

بر اساس جدول‌های بالا، پیشنهادها (پس از تأیید در کارت پروژه و فایل تنظیمات ثبت می‌شوند):

| تصمیم | پیشنهاد | دلیل |
|---|---|---|
| طول بازه | ۳ ماه (بدون تغییر) | تعادل بین به‌روز بودن و تعداد نقد؛ ۱ و ۶ ماه در فاز ۴ به‌عنوان تحلیل حساسیت |
| حداقل نقد در بازه | _بعد از دیدن جدول‌ها با هم تصمیم می‌گیریم_ | |
| شروع بازه‌ی تحلیل | _بعد از دیدن جدول سال‌ها_ | |
| پایان بازه‌ی تحلیل | _بعد از دیدن مقایسه‌ی ۲۰۲۳_ | |

## ۸. جمع‌بندی

_این بخش را بعد از تصمیم‌گیری با هم می‌نویسیم._

</div>